# 验证报告：4×4 行列式的列规范与行规范自由度

> 基于诊断报告 `logPsi_drift_diagnosis.md` 第 1 节的核心结论，用纯矩阵运算做数值验证。
>
> **模型设置**：取 $K=4$，用随机 $4\times4$ 复矩阵 $L$ 代表 `logψ_j(x_i)`，即 $L[i,j] = \log\psi_j(x_i)$。
>
> **验证目标**：
> 1. 列规范修复后，`log_det_centered` 对列平移严格不变。
> 2. 行规范未修复时，`log_det_centered` 会随行平移发生 $+\sum_i c_i$ 的漂移。
> 3. 对比"仅列修复"与"列+行双修复"对 drift 的影响。

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("NumPy 版本:", np.__version__)
print("验证环境已就绪")

NumPy 版本: 2.4.6
验证环境已就绪


---
## 1. 基础工具函数

模拟诊断报告中 `_stable_from_L` 和 `_compute_L_centered_single` 的 4×4 实现。

In [2]:
def stable_logdet(L):
    """
    对应 _stable_from_L 的逻辑（返回 log_det_centered）。
    L : (K, K) 复数矩阵
    shift = max(Re(L))
    Psi_stable = exp(L - shift)
    log_det_centered = slogdet(Psi_stable) + K * shift
    """
    shift = np.max(np.real(L))
    L_stable = L - shift
    Psi_stable = np.exp(L_stable)
    sign, log_abs_det = np.linalg.slogdet(Psi_stable)
    log_det_centered = log_abs_det + 1j * np.angle(sign) + len(L) * shift
    return log_det_centered, shift


def column_centered_L(L, ref_col):
    """
    对应 _compute_L_centered_single：每列减去参考态值
    ref_col : (K,) 参考态的各列 log 值
    """
    return L - ref_col[np.newaxis, :]


def make_ref_from_L(L):
    """用 L 的第0行作为参考态 logψ_j(ref)"""
    return L[0, :]


def row_centered_L(L):
    """
    行规范修复：每行减去该行均值（axis=1），使每行和为0
    """
    row_mean = np.mean(L, axis=1, keepdims=True)
    return L - row_mean

---
## 2. 生成随机 4×4 复矩阵 L₀

L₀[i,j] ≈ log ψ_j(x_i)，作为初始 ansatz 输出。

In [3]:
np.random.seed(20240824)
K = 4
L0 = np.random.randn(K, K) + 1j * np.random.randn(K, K)

print("初始 L0 (实部):")
print(np.round(np.real(L0), 4))
print("\n初始 L0 (虚部):")
print(np.round(np.imag(L0), 4))
print("\ndet(L0) =", np.linalg.det(L0))
print("cond(L0) =", np.linalg.cond(L0))

ref_col = make_ref_from_L(L0)
L_centered_base = column_centered_L(L0, ref_col)
log_det_base, shift_base = stable_logdet(L_centered_base)

print("\n--- 初始 L_centered（仅列修复） ---")
print("shift =", np.round(shift_base, 6))
print("log_det_centered =", np.round(log_det_base, 6))

初始 L0 (实部):
[[ 0.6698 -0.531  -1.1102 -1.0513]
 [ 0.4002  0.4829 -0.226  -0.7595]
 [-1.8202  0.0516 -0.0501 -0.8299]
 [ 0.8656 -0.7061 -1.0885 -0.4933]]

初始 L0 (虚部):
[[ 2.0863 -1.0231  0.4434 -0.8256]
 [-0.218   0.0628 -0.8453  0.5485]
 [-1.5797  0.6187  0.8927  1.0935]
 [-0.2718 -0.4554  0.7529 -0.6858]]

det(L0) = (-1.1806720660215158+0.19694236839404958j)
cond(L0) = 48.305894525349835

--- 初始 L_centered（仅列修复） ---
shift = 1.060045
log_det_centered = (2.825312+1.848995j)


---
## 3. 验证一：列规范不变性

**目标**：施加任意列平移 $L \to L + \mathbf{1}\cdot c^T$ 后，
`column_centered_L` 输出不变，因此 `log_det_centered` 也不变。

这是诊断报告 §1.2 的核心断言：**列规范被杀死**。

In [ ]:
c_col = np.array([1.5 + 0.3j, -0.8 + 0.1j, 2.0 - 0.5j, -1.2 + 0.7j])
L_col_gauged = L0 + c_col[np.newaxis, :]
L_centered_gauged = column_centered_L(L_col_gauged, ref_col)
log_det_gauged, shift_gauged = stable_logdet(L_centered_gauged)

print("【原始 L_centered】")
print("  实部第一行:", np.round(np.real(L_centered_base[0]), 4))
print("  shift     =", np.round(shift_base, 6))
print("  log_det   =", np.round(log_det_base, 6))

print("\n【施加列规范变换后 (c =", np.round(c_col, 2), ")】")
print("  实部第一行:", np.round(np.real(L_centered_gauged[0]), 4))
print("  shift     =", np.round(shift_gauged, 6))
print("  log_det   =", np.round(log_det_gauged, 6))

diff_col = np.abs(log_det_base - log_det_gauged)
print("\n*** 验证结果 ***")
print(f"  |log_det_原始 - log_det_规范变换| = {diff_col:.2e}")
if diff_col < 1e-10:
    print("  PASS: 列规范完全不变！log_det_centered 在列平移下严格保持。")
else:
    print("  FAIL: 发现显著差异，请检查。")

---
## 4. 验证二：行规范漂移 = Σcᵢ

**目标**：行规范变换 $L[i,j] \to L[i,j] + c_i$（每行加常数）后，
`column_centered_L` 不保持（因为 ref 只有一个样本），
且 `log_det_centered` 的实部变化恰好等于 $\sum_i c_i$。

这是诊断报告 §1.2–§1.3 的核心论断，解释了 `logΨ mean` 单调上升的原因。

In [ ]:
c_row_real = np.array([0.3, 0.5, 0.2, 0.4])  # Σc_i = 1.4
L_row_gauged = L0 + c_row_real[:, np.newaxis]

L_centered_rg = column_centered_L(L_row_gauged, ref_col)
log_det_rg, shift_rg = stable_logdet(L_centered_rg)

expected_drift = np.sum(c_row_real)
actual_drift = np.real(log_det_rg - log_det_base)

print("【施加实数行规范变换】")
print(f"  c_row = {c_row_real}")
print(f"  Σ c_i = {expected_drift:.6f}")
print(f"  log_det_原始           = {np.real(log_det_base):.6f}")
print(f"  log_det_行规范变换后   = {np.real(log_det_rg):.6f}")
print(f"  实际漂移 ΔRe           = {actual_drift:.6f}")
print(f"  理论漂移 Σc_i          = {expected_drift:.6f}")
print(f"  误差                   = {abs(actual_drift - expected_drift):.2e}")

if abs(actual_drift - expected_drift) < 1e-10:
    print("\n  PASS: 行规范漂移 = Σc_i 精确成立！")
else:
    print("\n  FAIL: 存在显著偏差。")

---
## 5. 验证三：多步 drift 累积（模拟训练过程）

假设每一步因 MC 噪声产生一个小量行偏移 $c_i^{(t)}$，观察 `log_det_centered` 的累积行为。

In [ ]:
np.random.seed(42)
n_steps = 200

c_per_step = np.random.normal(0.0, 0.02, size=(n_steps, K))
cumsum_c = np.cumsum(np.sum(c_per_step, axis=1))

L_current = L0.copy()
log_det_history = []

for t in range(n_steps):
    L_current = L_current + c_per_step[t, :, np.newaxis]
    L_centered_t = column_centered_L(L_current, ref_col)
    log_det_t, _ = stable_logdet(L_centered_t)
    log_det_history.append(np.real(log_det_t))

log_det_history = np.array(log_det_history)

fig, axes = plt.subplots(2, 1, figsize=(8, 6))

axes[0].plot(cumsum_c, 'b-', linewidth=1.5, label='理论累积 Σc_i')
axes[0].plot(log_det_history - log_det_history[0], 'r--', linewidth=1.5, label='实际 log_det 变化')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('log|det| change')
axes[0].set_title('行规范漂移累积：理论 vs 实际（K=4，200步）')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

final_theory = cumsum_c[-1]
final_actual = log_det_history[-1] - log_det_history[0]
axes[1].bar(['理论 Σc_i', '实际 Δlog|det|'], [final_theory, final_actual],
            color=['blue', 'red'], alpha=0.7)
axes[1].set_ylabel('Final value')
axes[1].set_title(f'最终漂移对比（误差 = {abs(final_theory - final_actual):.2e}）')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/yangjianfei/mac_vscode/神经网络量子态/NES_VMC/验证报告/row_gauge_drift_accumulation.png', dpi=150)
print("图表已保存至 row_gauge_drift_accumulation.png")
print(f"理论最终漂移: {final_theory:.4f}")
print(f"实际最终漂移: {final_actual:.4f}")
print(f"相对误差: {abs(final_theory - final_actual)/max(abs(final_theory),1e-10):.2%}")

---
## 6. 验证四：列+行双修复消除漂移

在列修复的基础上，进一步减去每行的均值（`row_centered_L`），锁定行规范自由度。

注：诊断报告 §5.1 中写的是 `axis=0`（沿列求均值），但那是杀列规范的方向；正确的行规范修复应沿 `axis=1`（沿 walker 维度求均值）。

In [ ]:
L_double = L0 + c_row_real[:, np.newaxis]

# 仅列修复
L_col_only = column_centered_L(L_double, ref_col)
log_det_col_only, _ = stable_logdet(L_col_only)

# 列 + 行双修复
L_both = row_centered_L(column_centered_L(L_double, ref_col))
log_det_both, _ = stable_logdet(L_both)

print("【对比：仅列修复 vs 列+行双修复】")
print(f"  仅列修复  log_det 实部 = {np.real(log_det_col_only):.6f}")
print(f"  列+行双修 log_det 实部 = {np.real(log_det_both):.6f}")
print(f"  差异 = {np.real(log_det_col_only - log_det_both):.6f}  (应 ≈ Σc_i = {expected_drift})")

extra_row_shift = np.array([0.5, 0.3, 0.7, 0.2])
L_after_extra = row_centered_L(column_centered_L(L0 + extra_row_shift[:, np.newaxis], ref_col))
log_det_extra, _ = stable_logdet(L_after_extra)

diff_after_fix = np.abs(np.real(log_det_both - log_det_extra))
print(f"\n【双修复后再施加行规范变换】")
print(f"  |log_det_双修复 - log_det_再加行规范| = {diff_after_fix:.2e}")
if diff_after_fix < 1e-10:
    print("  PASS: 双修复后 log_det 对行规范变换严格不变！")
else:
    print("  FAIL: 仍有残余漂移。")

---
## 7. 汇总表

汇总所有验证结果，与诊断报告各节对照。

In [ ]:
results = {
    '列规范不变性': diff_col < 1e-10,
    '行规范漂移 = Σc_i': abs(actual_drift - expected_drift) < 1e-10,
    '双修复消除行规范漂移': diff_after_fix < 1e-10,
}

print('=' * 55)
print('验证结果汇总')
print('=' * 55)
for name, passed in results.items():
    status = 'PASS' if passed else 'FAIL'
    mark = '[OK]' if passed else '[!!]'
    print(f'  {mark} {status}: {name}')
print('=' * 55)

all_pass = all(results.values())
print(f"\n全部验证 {'通过' if all_pass else '未通过'}！\n")
print('与诊断报告的对应关系：')
print('  • 列规范不变    → 确认 §1.2 "列规范被杀死" 的结论')
print('  • 行规范漂移    → 确认 §1.2 的数学推导：logΨ mean 漂移 ≈ Σc_i')
print('  • 双修复消除    → 确认 §5.1 建议的行规范修复方向是正确的')
print('\n注意：诊断报告 §5.1 中的 axis=0 有误，正确应为 axis=1（沿 walker 维度求均值）。')